# Model Serialization

This notebook demonstrates saving and loading models in NeuraCommand.

**Key features:**
- Safe format: JSON + NPZ (no pickle!)
- Human-readable architecture file
- Portable across Python versions
- Metadata support

In [ ]:
import numpy as np
import json
import os
import tempfile
import sys
sys.path.insert(0, '../src')

from neura_command import (
    Sequential, Dense, ReLU, Sigmoid, Dropout, BatchNorm, Softmax,
    Adam, SGD, MSELoss, CrossEntropyLoss,
    save_model, load_model, get_model_info,
)

np.random.seed(42)

## 1. Basic Save and Load

In [ ]:
# Create a model
model = Sequential(
    layers=[
        Dense(10, 32),
        ReLU(),
        Dropout(0.2),
        Dense(32, 16),
        ReLU(),
        Dense(16, 2),
        Softmax(),
    ],
    optimizer=Adam(learning_rate=0.001),
    loss=CrossEntropyLoss(),
)

print("Original model:")
print(model.summary())

In [ ]:
# Create a temporary directory for saving
tmpdir = tempfile.mkdtemp()
model_path = os.path.join(tmpdir, 'my_model')

# Save the model
model.save(model_path)

print(f"Model saved to: {model_path}")
print(f"\nCreated files:")
for f in os.listdir(tmpdir):
    size = os.path.getsize(os.path.join(tmpdir, f))
    print(f"  {f}: {size:,} bytes")

In [ ]:
# Load the model
loaded_model = Sequential.load(model_path)

print("Loaded model:")
print(loaded_model.summary())

In [ ]:
# Verify predictions match
X_test = np.random.randn(5, 10).astype(np.float32)

original_preds = model.predict(X_test)
loaded_preds = loaded_model.predict(X_test)

print("Predictions match:", np.allclose(original_preds, loaded_preds))
print("\nOriginal predictions:")
print(original_preds)
print("\nLoaded model predictions:")
print(loaded_preds)

## 2. Inspect the Saved Files

In [ ]:
# Look at the JSON architecture file
with open(model_path + '.json', 'r') as f:
    arch = json.load(f)

print("Architecture file contents:")
print(json.dumps(arch, indent=2))

In [ ]:
# Look at the weights file
weights = np.load(model_path + '.npz')

print("Weights file contents:")
for key in weights.files:
    arr = weights[key]
    print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")

## 3. Get Model Info

In [ ]:
# Use get_model_info() to inspect a saved model without loading it
info = get_model_info(model_path)

print("Model Info:")
print("-" * 40)
print(f"Number of layers: {info['num_layers']}")
print(f"Layer types: {info['layers']}")
print(f"Optimizer: {info['optimizer']}")
print(f"Loss: {info['loss']}")
print(f"Weights size: {info['weights_size_bytes'] / 1024:.1f} KB")
print(f"NeuraCommand version: {info['neura_command_version']}")

## 4. Save with Custom Metadata

In [ ]:
# Train a small model
X_train = np.random.randn(100, 10).astype(np.float32)
y_train = np.random.randint(0, 2, (100, 2)).astype(np.float32)

model = Sequential(
    layers=[Dense(10, 16), ReLU(), Dense(16, 2), Softmax()],
    optimizer=Adam(learning_rate=0.01),
    loss=CrossEntropyLoss(),
)

model.train(X_train, y_train, epochs=10, batch_size=16, verbose=False)

In [ ]:
# Save with metadata
model_path2 = os.path.join(tmpdir, 'model_with_metadata')

save_model(model, model_path2, metadata={
    'description': 'Binary classification model',
    'author': 'NeuraCommand Demo',
    'training_accuracy': 0.85,
    'epochs_trained': 10,
    'dataset': 'synthetic',
})

print("Model saved with metadata!")

In [ ]:
# Retrieve the metadata
info = get_model_info(model_path2)

print("Saved metadata:")
for key, value in info['metadata'].items():
    print(f"  {key}: {value}")

## 5. Save and Load Different Layer Types

In [ ]:
# Model with BatchNorm
model_bn = Sequential(
    layers=[
        Dense(10, 32),
        BatchNorm(32),
        ReLU(),
        Dense(32, 2),
    ],
    optimizer=Adam(),
    loss=MSELoss(),
)

# Do a forward pass to initialize BatchNorm running stats
X_dummy = np.random.randn(32, 10).astype(np.float32)
_ = model_bn.forward(X_dummy, training=True)

print("BatchNorm running mean:", model_bn.layers[1].running_mean)

In [ ]:
# Save and load
model_path3 = os.path.join(tmpdir, 'model_batchnorm')
model_bn.save(model_path3)

loaded_bn = Sequential.load(model_path3)

print("Loaded BatchNorm running mean:", loaded_bn.layers[1].running_mean)
print("\nRunning stats preserved:", 
      np.allclose(model_bn.layers[1].running_mean, loaded_bn.layers[1].running_mean))

## 6. Save and Load with Different Optimizers

In [ ]:
# Model with SGD (momentum)
model_sgd = Sequential(
    layers=[Dense(5, 3)],
    optimizer=SGD(learning_rate=0.1, momentum=0.9),
    loss=MSELoss(),
)

model_path4 = os.path.join(tmpdir, 'model_sgd')
model_sgd.save(model_path4)

loaded_sgd = Sequential.load(model_path4)

print(f"Original optimizer: {model_sgd.optimizer.__class__.__name__}")
print(f"  learning_rate: {model_sgd.optimizer.learning_rate}")
print(f"  momentum: {model_sgd.optimizer.momentum}")

print(f"\nLoaded optimizer: {loaded_sgd.optimizer.__class__.__name__}")
print(f"  learning_rate: {loaded_sgd.optimizer.learning_rate}")
print(f"  momentum: {loaded_sgd.optimizer.momentum}")

## 7. Trained Model Round-trip

In [ ]:
# Create and train a model
X_train = np.random.randn(200, 4).astype(np.float32)
y_train = (X_train[:, 0] + X_train[:, 1] > 0).astype(np.float32).reshape(-1, 1)

model = Sequential(
    layers=[Dense(4, 16), ReLU(), Dense(16, 1), Sigmoid()],
    optimizer=Adam(learning_rate=0.01),
    loss=MSELoss(),
)

# Train
model.train(X_train, y_train, epochs=50, batch_size=32, verbose=False)

# Evaluate
preds_before = model.predict(X_train)
acc_before = np.mean((preds_before > 0.5) == y_train) * 100
print(f"Accuracy before saving: {acc_before:.1f}%")

In [ ]:
# Save, load, and verify
model_path5 = os.path.join(tmpdir, 'trained_model')
model.save(model_path5)

loaded = Sequential.load(model_path5)

preds_after = loaded.predict(X_train)
acc_after = np.mean((preds_after > 0.5) == y_train) * 100
print(f"Accuracy after loading: {acc_after:.1f}%")

print(f"\nPredictions identical: {np.allclose(preds_before, preds_after)}")

## 8. Continue Training After Loading

In [ ]:
# Load model and continue training
loaded = Sequential.load(model_path5)

print(f"Before continued training: {acc_after:.1f}%")

# Train more
loaded.train(X_train, y_train, epochs=50, batch_size=32, verbose=False)

# Evaluate
preds_continued = loaded.predict(X_train)
acc_continued = np.mean((preds_continued > 0.5) == y_train) * 100
print(f"After continued training: {acc_continued:.1f}%")

## 9. Cleanup

In [ ]:
import shutil
shutil.rmtree(tmpdir)
print("Temporary files cleaned up!")

## Summary

NeuraCommand serialization features:

| Feature | Description |
|---------|-------------|
| **Format** | JSON (architecture) + NPZ (weights) |
| **Security** | No pickle - safe to load untrusted models |
| **Portability** | Works across Python versions |
| **Inspection** | `get_model_info()` without loading |
| **Metadata** | Store training info, author, etc. |
| **Layers** | Dense, Dropout, BatchNorm, all activations |
| **Optimizers** | Adam, SGD, RMSprop, AdaGrad |
| **BatchNorm** | Running stats preserved |

**Usage:**
```python
# Save
model.save('path/to/model')

# Load
model = Sequential.load('path/to/model')

# Inspect
info = get_model_info('path/to/model')
```